In [1]:
# PHASE 2 - ABLATION 3 (Method 2 / CCSM, Lunit ViT-S/16): is the Fourier amplitude swap alone enough?  Seeds 42/43/44.
# Kaggle: GPU on, Internet on.  Attach NOTHING for a first run.  If the session stops at the time limit, Save Version and re-run with that OUTPUT attached as an input.
# Arm: SimCLR + Fourier amplitude swap (window 0.05, cross-centre partner), NO MixStyle.  Same pool / augmentation / schedule as the main Lunit CCSM run (control, ms, fms are prefilled).
# Pipeline per seed: pretrain (~13 min) -> extract features (~25 min) -> frozen linear probe.  Estimated runtime ~2-2.5 h.
!pip install -q timm datasets transformers kagglehub

"""
Frozen linear probe, Part-I recipe (StandardScaler + LogisticRegression C=1 balanced, max_iter 2000; primary split centres 0,1,3,4 -> 2 with 1000
patient-level bootstraps, seed 7; 5-fold leave-one-centre-out; patch-level centre-ID probe). Reference rows are PREFILLED from the earlier Lunit evaluations.
Caveat: for LOCO folds 0,1,3,4 the held-out centre's UNLABELED images were in the pretraining pool; centre 2 (primary split) is clean.
"""
import json, glob, time, pandas as pd
import os, time, random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore')

FAMILY = 'lunit'
SPECS = {   # kind, hub, input size, patches per centre, batch (pairs), epochs
    'vit_small': dict(kind='timm', hub='vit_small_patch16_224', size=224, per_center=5000, batch=64, epochs=6),
    'swin_tiny': dict(kind='timm', hub='swin_tiny_patch4_window7_224', size=224, per_center=5000, batch=64, epochs=6),
    'kaiko': dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms', size=224, per_center=5000, batch=64, epochs=6),
    'lunit': dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.lunit_dino', size=224, per_center=5000, batch=64, epochs=6),
    'phikon': dict(kind='hf', hub='owkin/phikon', size=224, per_center=3000, batch=64, epochs=6),
}
SP = SPECS[FAMILY]

class Cfg:
    HF = "wltjr1007/Camelyon17-WILDS"
    SEEDS = [42, 43, 44]
    POOL_SEED = 42
    PRE_CENTERS = [0, 1, 3, 4]
    N_PER_CENTER = SP['per_center']
    EPOCHS = SP['epochs']; BATCH = SP['batch']; LR = 3e-4; BB_MULT = 0.1; WD = 0.05; TAU = 0.1
    FOURIER_BETA = 0.05; LAM_MAX = 1.0
    MS_P = 0.5; MS_ALPHA = 0.1
    ARMS = ['fonly', 'msrand']
    LOAD_SIZE = 96
    SIZE = SP['size']
    CKPT_DIR = "/kaggle/working/checkpoints"
    PRETRAINED = True

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def load_pool(hf):
    from tqdm.auto import tqdm
    centers = np.array(hf['center'])
    rng = np.random.RandomState(Cfg.POOL_SEED); idx = []
    for c in Cfg.PRE_CENTERS:
        idx += list(rng.choice(np.where(centers == c)[0], Cfg.N_PER_CENTER, replace=False))
    imgs = np.zeros((len(idx), Cfg.LOAD_SIZE, Cfg.LOAD_SIZE, 3), np.uint8)
    for k, i in enumerate(tqdm(idx, desc='loading pool')):
        im = hf[int(i)]['image'].convert('RGB')
        if im.size != (Cfg.LOAD_SIZE, Cfg.LOAD_SIZE): im = im.resize((Cfg.LOAD_SIZE, Cfg.LOAD_SIZE))
        imgs[k] = np.asarray(im)
    return torch.from_numpy(imgs).permute(0, 3, 1, 2).contiguous(), torch.tensor(centers[idx])

def base_aug(x):
    B = x.size(0)
    a = torch.empty(B, device=x.device).uniform_(0.4, 1.0); s = a.sqrt()
    tx = (torch.rand(B, device=x.device) * 2 - 1) * (1 - s); ty = (torch.rand(B, device=x.device) * 2 - 1) * (1 - s)
    fx = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0); fy = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0)
    th = torch.zeros(B, 2, 3, device=x.device); th[:, 0, 0] = s * fx; th[:, 1, 1] = s * fy; th[:, 0, 2] = tx; th[:, 1, 2] = ty
    x = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode='reflection', align_corners=False)
    r = lambda lo, hi: torch.empty(B, 1, 1, 1, device=x.device).uniform_(lo, hi)
    x = x * r(0.8, 1.2)
    m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * r(0.8, 1.2) + m
    g = x.mean(1, keepdim=True); x = (x - g) * r(0.8, 1.2) + g
    return x.clamp(0, 1)

def partner_index(c, cross=True):
    if not cross:
        return torch.randperm(len(c), device=c.device)          # partner drawn at random (may be the same centre)
    mask = (c[:, None] != c[None, :]).float()
    mask[mask.sum(1) == 0] = 1.0
    return torch.multinomial(mask, 1).squeeze(1)

def fourier_swap(x, j, beta, lam_max):
    B, _, H, W = x.shape
    Fx = torch.fft.fft2(x.float()); amp = torch.fft.fftshift(Fx.abs(), dim=(-2, -1)); ph = Fx.angle()
    h = int(beta * H); c = H // 2; sl = slice(c - h, c + h + 1)
    lam = torch.empty(B, 1, 1, 1, device=x.device).uniform_(0, lam_max)
    new = amp.clone()
    new[..., sl, sl] = (1 - lam) * amp[..., sl, sl] + lam * amp[j][..., sl, sl]
    new = torch.fft.ifftshift(new, dim=(-2, -1))
    return torch.fft.ifft2(new * torch.exp(1j * ph)).real.clamp(0, 1)

class PhikonBackbone(nn.Module):
    """owkin/phikon -> CLS token (identical to Part I and to the adjacency E notebooks)."""
    def __init__(self, hub, pretrained=True):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        self.vit = AutoModel.from_pretrained(hub) if pretrained else AutoModel.from_config(AutoConfig.from_pretrained(hub))
        self.dim = self.vit.config.hidden_size
    def forward(self, x):
        return self.vit(pixel_values=x).last_hidden_state[:, 0]

def make_backbone():
    if SP['kind'] == 'hf':
        bb = PhikonBackbone(SP['hub'], Cfg.PRETRAINED)
        bb.vit.gradient_checkpointing_enable()
        first = bb.vit.encoder.layer[0] if hasattr(bb.vit, 'encoder') else bb.vit.layers[0]      # transformers 4.x / 5.x
        return bb, bb.dim, first
    import timm
    bb = timm.create_model(SP['hub'], pretrained=Cfg.PRETRAINED, num_classes=0)
    dim = bb.num_features
    target = bb.blocks[0] if hasattr(bb, 'blocks') else bb.layers[0]
    return bb, dim, target

class Net(nn.Module):
    """MixStyle is applied by a forward hook on the first block/stage. The mixing decision (on/off, lambda, partner) is drawn ONCE per
    step outside the forward pass, so it is identical under gradient-checkpoint recomputation."""
    def __init__(self, mixstyle=False):
        super().__init__()
        self.backbone, dim, target = make_backbone()
        self.head = nn.Sequential(nn.Linear(dim, 512), nn.ReLU(), nn.Linear(512, 128))
        self.mixstyle = mixstyle; self.mix = None          # (lam [B,1,..], partner idx) or None
        if mixstyle:
            target.register_forward_hook(self._hook)
    def _hook(self, mod, inp, out):
        if not (self.training and self.mix is not None):
            return out
        h = out[0] if isinstance(out, tuple) else out
        dims = (1,) if h.dim() == 3 else (1, 2)            # tokens [B,N,C] or NHWC [B,H,W,C]
        lam, j = self.mix
        lam = lam.view(-1, *([1] * (h.dim() - 1))).to(h.dtype)
        mu = h.mean(dims, keepdim=True); sd = (h.var(dims, keepdim=True) + 1e-6).sqrt()
        h2 = (h - mu) / sd * (lam * sd + (1 - lam) * sd[j]) + (lam * mu + (1 - lam) * mu[j])
        return (h2,) + tuple(out[1:]) if isinstance(out, tuple) else h2
    def forward(self, x):
        return F.normalize(self.head(self.backbone(x)), dim=1)

def nt_xent(z, tau):
    n = z.size(0) // 2
    sim = z @ z.t() / tau; sim.fill_diagonal_(-1e9)
    tgt = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, tgt), (sim.argmax(1) == tgt).float().mean().item()

def prep(x):
    if x.shape[-1] != Cfg.SIZE:
        x = F.interpolate(x, size=(Cfg.SIZE, Cfg.SIZE), mode='bilinear', align_corners=False)
    return (x - MEAN.to(x.device)) / STD.to(x.device)

def pretrain(arm, imgs, cen, seed):
    set_seed(seed)
    ms = arm in ('ms', 'msrand', 'fms', 'fmsrand'); use_f = arm in ('fms', 'fonly', 'fmsrand'); cross = arm not in ('msrand', 'fmsrand')
    net = Net(mixstyle=ms).to(dev).train()
    opt = torch.optim.AdamW([{'params': net.backbone.parameters(), 'lr': Cfg.LR * Cfg.BB_MULT},
                             {'params': net.head.parameters(), 'lr': Cfg.LR}], weight_decay=Cfg.WD)
    spe = len(imgs) // Cfg.BATCH
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, [Cfg.LR * Cfg.BB_MULT, Cfg.LR], total_steps=Cfg.EPOCHS * spe, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=dev.type == 'cuda'); chance = float(np.log(2 * Cfg.BATCH - 1))
    for ep in range(Cfg.EPOCHS):
        perm = torch.randperm(len(imgs)); L, A, t0 = 0.0, 0.0, time.time()
        for b in range(spe):
            ix = perm[b * Cfg.BATCH:(b + 1) * Cfg.BATCH]
            x = imgs[ix].to(dev).float() / 255; c = cen[ix].to(dev)
            v1, v2 = base_aug(x), base_aug(x)
            if use_f:
                v2 = fourier_swap(v2, partner_index(c, cross), Cfg.FOURIER_BETA, Cfg.LAM_MAX)
            if ms:
                if random.random() < Cfg.MS_P:
                    cc = torch.cat([c, c]); lam = torch.distributions.Beta(Cfg.MS_ALPHA, Cfg.MS_ALPHA).sample((cc.numel(),)).to(dev)
                    net.mix = (lam, partner_index(cc, cross))
                else:
                    net.mix = None
            with torch.autocast(dev.type, enabled=dev.type == 'cuda'):
                z = net(prep(torch.cat([v1, v2])))
            loss, acc = nt_xent(z.float(), Cfg.TAU)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            L += loss.item(); A += acc
        print(f"  [{arm} s{seed}] ep {ep + 1}/{Cfg.EPOCHS} loss {L / spe:.3f} (chance {chance:.2f}) inst-acc {A / spe:.3f} | {(time.time() - t0) / 60:.1f} min")
    if L / spe > 0.95 * chance:
        print(f"  !! [{arm}] loss still at chance -> do NOT use this arm")
    net.mix = None
    return net.eval(), L / spe

def ccsm_train_main(hf=None, tensors=None):
    imgs, cen = tensors if tensors is not None else load_pool(hf)
    os.makedirs(Cfg.CKPT_DIR, exist_ok=True)
    for seed in Cfg.SEEDS:
        for arm in Cfg.ARMS:
            path = f"{Cfg.CKPT_DIR}/ccsm_{FAMILY}_{arm}_s{seed}_best.pt"
            if os.path.exists(path):
                print(f"  [{arm} s{seed}] already done -> skip"); continue
            t = time.time(); net, L = pretrain(arm, imgs, cen, seed)
            torch.save({'model_state_dict': net.state_dict(), 'arm': arm, 'seed': seed, 'epoch': Cfg.EPOCHS, 'best_val': L}, path)
            print(f"  [{arm} s{seed}] saved {path} | {(time.time() - t) / 60:.1f} min")
    print("\nDONE. Files:", sorted(os.listdir(Cfg.CKPT_DIR)))



# ---- frozen linear-probe evaluation (same code as the D7/D8 evaluations) ----
import os, gc, json, glob, random, time, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm import tqdm

warnings.filterwarnings('ignore')


class EC:
    RUN_FAMILIES = ['lunit']
    SEEDS = [42, 43, 44]
    BUDGET_H = 11.0                       # stop starting new chunks when the next one would not fit
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CKPT_SEARCH_ROOTS = ["/kaggle/working/checkpoints"]
    KAGGLE_CKPT_DATASETS = ["sethu2954/adjc-kaiko-checkpoints", "sethu2954/adjc-lunit-checkpoints", "sethu2954/adjc-phikon-checkpoints"]   # fallback if not attached
    PRIOR_RESULT_ROOTS = ["/kaggle/input"]                                  # earlier runs' outputs (p2_eval_all/*.json) are reused
    FEATURE_DIR = "/kaggle/working/abl_features"
    RESULTS_DIR = "/kaggle/working/abl_eval"
    KEEP_FEATURES = False

    EXTRACT_BATCH = 128
    NUM_WORKERS = 4
    ALL_CENTERS = [0, 1, 2, 3, 4]
    TEST_CENTER = 2
    LR_C, LR_MAX_ITER, PROBE_SEED = 1.0, 2000, 42
    BOOT_N, BOOT_SEED = 1000, 7
    LOCO_FOLDS = [0, 1, 2, 3, 4]
    CID_TRAIN_PER_CENTER, CID_TEST_PER_CENTER, CID_VAL_FRAC = 3000, 1000, 0.25

    FAMILIES = {   # kind, hub id / timm name, input size, fp16 extraction, Part-I LOCO of the original (pipeline check; None = ImageNet control)
        'resnet18':  dict(kind='resnet', hub=None, size=96,  amp=False, ref=0.9528, label='ImageNet ResNet18'),
        'vit_small': dict(kind='timm', hub='vit_small_patch16_224', size=224, amp=False, ref=None, label='ImageNet ViT-S/16'),
        'swin_tiny': dict(kind='timm', hub='swin_tiny_patch4_window7_224', size=224, amp=False, ref=None, label='ImageNet Swin-T'),
        'kaiko':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms', size=224, amp=False, ref=0.9873, label='Kaiko ViT-S/16'),
        'lunit':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.lunit_dino', size=224, amp=False, ref=0.9814, label='Lunit ViT-S/16'),
        'phikon':    dict(kind='hf', hub='owkin/phikon', size=224, amp=True, ref=0.9818, label='Phikon ViT-B/16'),
    }
    ARMS = {}
    DEFAULT_ARMS = ['neighbor', 'self']
    BACKBONE = {'kaiko': 'vit_small', 'lunit': 'vit_small', 'phikon': 'phikon'}      # name used in the checkpoint file names
    PART1_OTHERS = "Part-I LOCO references: Stain Aug 0.9884 | Kaiko 0.9873 | Phikon 0.9818 | Lunit 0.9814 | ImageNet ResNet18 0.9528"

    @staticmethod
    def arms(fam):
        return EC.ARMS.get(fam, EC.DEFAULT_ARMS)

    @staticmethod
    def chunks(fam):
        """[(chunk_models)] : chunk 1 = original + seed 42, then one chunk per further seed.  Model name = '<family>__ctrl' / '<family>__<arm>_s<seed>'."""
        out = []
        for k, s in enumerate(EC.SEEDS):
            c = [f'{fam}__ctrl'] if k == 0 else []
            c += [f'{fam}__{a}_s{s}' for a in EC.arms(fam)]
            out.append(c)
        return out


def parse(name):
    fam, rest = name.split('__')
    if rest == 'ctrl':
        return fam, 'ctrl', None
    arm, s = rest.rsplit('_s', 1)
    return fam, arm, int(s)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ---------------------------------------------------------------- encoders (same construction as Part I / the training notebooks)
class Encoder(nn.Module):
    def __init__(self, fam):
        super().__init__()
        spec = EC.FAMILIES[fam]; self.pool = None
        if spec['kind'] == 'resnet':
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.backbone = nn.Sequential(*list(net.children())[:-1]); self.pool = nn.AdaptiveAvgPool2d(1); self.dim = 512
        elif spec['kind'] == 'hf':
            from transformers import AutoModel
            class PhikonBackbone(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.vit = AutoModel.from_pretrained(spec['hub'])
                def forward(self, x):
                    return self.vit(pixel_values=x).last_hidden_state[:, 0]           # CLS token, as in Part I
            self.backbone = PhikonBackbone(); self.dim = self.backbone.vit.config.hidden_size
        else:
            import timm
            self.backbone = timm.create_model(spec['hub'], pretrained=True, num_classes=0)
            self.dim = self.backbone.num_features

    @torch.no_grad()
    def encode(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f


def ensure_checkpoint_roots():
    if glob.glob("/kaggle/input/**/adjclv4-*_best.pt", recursive=True):
        return
    import kagglehub
    for slug in EC.KAGGLE_CKPT_DATASETS:
        try:
            path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", path)
            EC.CKPT_SEARCH_ROOTS.append(path)
        except Exception as e:
            print(f"⚠ could not download {slug}: {e}")


def find_checkpoint(fam, arm, seed):
    fname = f"adjclv4-{fam}_{EC.BACKBONE[fam]}_{arm}_s{seed}_best.pt"
    for root in EC.CKPT_SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits:
            return hits[0]
    return None


def build_model(name):
    fam, arm, seed = parse(name)
    if arm == 'ctrl':
        return Encoder(fam)
    path = find_checkpoint(fam, arm, seed)
    if path is None:
        return None
    enc = Encoder(fam)
    ck = torch.load(path, map_location='cpu')
    missing, unexpected = enc.load_state_dict(ck['model_state_dict'], strict=False)
    bad = [k for k in unexpected if not k.startswith(('head.', 'adv_head.'))]
    assert not missing and not bad, f"{name}: missing={missing[:3]} unexpected={bad[:3]}"
    print(f"  [{name}] loaded {os.path.basename(path)}  (epoch {ck.get('epoch','?')}, final loss {ck.get('best_val', float('nan')):.4f})")
    return enc


# ---------------------------------------------------------------- one-pass feature extraction
class MultiSizeDataset(Dataset):
    def __init__(self, hf, sizes):
        self.hf, self.sizes = hf, sorted(sizes)
        norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        self.tf = {s: transforms.Compose([transforms.Resize((s, s)), transforms.ToTensor(), norm]) for s in self.sizes}

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, i):
        img = self.hf[int(i)]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {s: self.tf[s](img) for s in self.sizes}


def feat_path(name):
    return f"{EC.FEATURE_DIR}/{name}.npy"


def extract_missing(hf, names, device):
    todo = [n for n in names if not os.path.exists(feat_path(n))]
    if not todo:
        return
    mods = {}
    for n in todo:
        m = build_model(n)
        if m is None:
            print(f"  [{n}] checkpoint not found -> skipped")
            continue
        mods[n] = m.to(device).eval()
    if not mods:
        return
    fam = parse(todo[0])[0]; size = EC.FAMILIES[fam]['size']; amp = EC.FAMILIES[fam]['amp'] and device.type == 'cuda'
    N = len(hf)
    out = {n: np.empty((N, mods[n].dim), dtype=np.float32) for n in mods}
    loader = DataLoader(MultiSizeDataset(hf, {size}), batch_size=EC.EXTRACT_BATCH, shuffle=False,
                        num_workers=EC.NUM_WORKERS, pin_memory=device.type == 'cuda')
    pos, t0 = 0, time.time()
    for batch in tqdm(loader, desc=f"extract {fam}: {len(mods)} models"):
        x = batch[size].to(device, non_blocking=True); B = x.size(0)
        for n, m in mods.items():
            with torch.autocast('cuda', enabled=amp):
                f = m.encode(x)
            out[n][pos:pos + B] = f.float().cpu().numpy()
        pos += B
    assert pos == N, (pos, N)
    os.makedirs(EC.FEATURE_DIR, exist_ok=True)
    for n in mods:
        np.save(feat_path(n), out[n])
    print(f"✓ extracted {list(mods)} in {(time.time()-t0)/60:.1f} min")
    del mods, out; gc.collect(); torch.cuda.empty_cache()


# ---------------------------------------------------------------- evaluation (same recipe as Part I)
def bootstrap_auroc_by_patient(y, p, pat, n_boot, seed):
    pats = np.unique(pat)
    idx_by = {q: np.where(pat == q)[0] for q in pats}
    rng, vals = np.random.RandomState(seed), []
    for _ in range(n_boot):
        s = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by[q] for q in s])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def probe_auroc(X, y, centers, patients, train_c, test_c, with_ci=False):
    tr, te = np.isin(centers, train_c), centers == test_c
    sc = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=EC.LR_MAX_ITER, C=EC.LR_C, class_weight='balanced',
                             random_state=EC.PROBE_SEED, n_jobs=-1)
    clf.fit(sc.transform(X[tr]), y[tr])
    prob = clf.predict_proba(sc.transform(X[te]))[:, 1]
    auc = roc_auc_score(y[te], prob)
    bal = balanced_accuracy_score(y[te], prob > 0.5)
    if with_ci:
        lo, hi = bootstrap_auroc_by_patient(y[te], prob, patients[te], EC.BOOT_N, EC.BOOT_SEED)
        return auc, bal, lo, hi
    return auc, bal


def center_id_probe(X, centers, patients):
    """Patch-level 5-way center-identity probe, patient-disjoint train/test within each center."""
    rng = np.random.RandomState(EC.PROBE_SEED)
    tr_idx, te_idx = [], []
    for c in EC.ALL_CENTERS:
        pats = sorted(np.unique(patients[centers == c]).tolist())
        random.Random(EC.PROBE_SEED + c).shuffle(pats)
        n_te = min(max(1, int(round(EC.CID_VAL_FRAC * len(pats)))), len(pats) - 1)
        te_p, tr_p = set(pats[:n_te]), set(pats[n_te:])
        for pool, n_want, store in ((np.where((centers == c) & np.isin(patients, list(tr_p)))[0], EC.CID_TRAIN_PER_CENTER, tr_idx),
                                    (np.where((centers == c) & np.isin(patients, list(te_p)))[0], EC.CID_TEST_PER_CENTER, te_idx)):
            store.append(rng.choice(pool, size=min(n_want, len(pool)), replace=False))
    tr_idx, te_idx = np.concatenate(tr_idx), np.concatenate(te_idx)
    sc = StandardScaler().fit(X[tr_idx])
    clf = LogisticRegression(max_iter=EC.LR_MAX_ITER, n_jobs=-1).fit(sc.transform(X[tr_idx]), centers[tr_idx])
    return 100 * accuracy_score(centers[te_idx], clf.predict(sc.transform(X[te_idx])))


def evaluate_model(name, y, centers, patients):
    cache = f"{EC.RESULTS_DIR}/{name}.json"
    if os.path.exists(cache):
        return json.load(open(cache))
    X = np.load(feat_path(name), mmap_mode='r')
    X = np.ascontiguousarray(X)
    t0 = time.time()
    auc, bal, lo, hi = probe_auroc(X, y, centers, patients, [c for c in EC.ALL_CENTERS if c != EC.TEST_CENTER],
                                   EC.TEST_CENTER, with_ci=True)
    loco = {}
    for ho in EC.LOCO_FOLDS:
        loco[ho] = probe_auroc(X, y, centers, patients, [c for c in EC.ALL_CENTERS if c != ho], ho)[0]
    vals = np.array(list(loco.values()))
    res = {'model': name, 'primary_auroc': auc, 'primary_ci_lo': lo, 'primary_ci_hi': hi, 'primary_balacc': bal,
           'loco_mean': float(vals.mean()), 'loco_std': float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
           'loco_folds': {str(k): float(v) for k, v in loco.items()},
           'center_id_acc': center_id_probe(X, centers, patients)}
    print(f"  [{name}] primary {auc:.4f} [{lo:.4f},{hi:.4f}] | LOCO {res['loco_mean']:.4f}±{res['loco_std']:.4f} | "
          f"center-ID {res['center_id_acc']:.1f}% | {(time.time()-t0)/60:.1f} min")
    os.makedirs(EC.RESULTS_DIR, exist_ok=True)
    json.dump(res, open(cache, 'w'))
    del X; gc.collect()
    if not EC.KEEP_FEATURES and os.path.exists(feat_path(name)):
        os.remove(feat_path(name))
    return res




# ====================================================================
# ABLATION SETUP
# ====================================================================
SEEDS = [42, 43, 44]
BUDGET_H = 11.0
TITLE = 'ABLATION 3 - Fourier amplitude swap only, no MixStyle (CCSM recipe)'
ARM_LABELS = {
 "fonly": "Fourier-only (window 0.05, no MixStyle)"
}
ARMS = list(ARM_LABELS)
PAIRS = [["fonly", "fms", "Fourier-only minus MixStyle+Fourier"], ["fonly", "ms", "Fourier-only minus MixStyle-only"], ["fonly", "control", "Fourier-only minus SimCLR control"]]
EC.SEEDS = SEEDS; EC.BUDGET_H = BUDGET_H
Cfg.ARMS = ARMS


REFERENCE = {   # earlier evaluations of the SAME Lunit ViT-S/16 models; name -> (primary, ci_lo, ci_hi, loco_mean, loco_std, centre_id)
 'lunit__ctrl':        (0.9783, 0.9620, 0.9933, 0.9814, 0.0116, 97.7),
 'lunit__control_s42': (0.9861, 0.9667, 0.9941, 0.9846, 0.0055, 97.4),
 'lunit__control_s43': (0.9832, 0.9588, 0.9931, 0.9825, 0.0073, 98.0),
 'lunit__control_s44': (0.9840, 0.9659, 0.9931, 0.9815, 0.0060, 97.9),
 'lunit__ms_s42':      (0.9873, 0.9720, 0.9937, 0.9870, 0.0040, 97.5),
 'lunit__ms_s43':      (0.9818, 0.9501, 0.9924, 0.9861, 0.0050, 97.6),
 'lunit__ms_s44':      (0.9871, 0.9751, 0.9934, 0.9857, 0.0051, 97.6),
 'lunit__fms_s42':     (0.9873, 0.9774, 0.9909, 0.9870, 0.0042, 94.5),
 'lunit__fms_s43':     (0.9874, 0.9760, 0.9917, 0.9864, 0.0044, 94.6),
 'lunit__fms_s44':     (0.9880, 0.9777, 0.9912, 0.9872, 0.0047, 95.3),
 'lunit__neighbor_s42': (0.9921, 0.9826, 0.9954, 0.9889, 0.0037, 89.5),
 'lunit__neighbor_s43': (0.9919, 0.9851, 0.9949, 0.9897, 0.0038, 90.2),
 'lunit__neighbor_s44': (0.9925, 0.9850, 0.9952, 0.9913, 0.0023, 91.4),
 'lunit__self_s42':     (0.9853, 0.9718, 0.9900, 0.9804, 0.0044, 86.2),
 'lunit__self_s43':     (0.9860, 0.9722, 0.9907, 0.9801, 0.0055, 86.5),
 'lunit__self_s44':     (0.9862, 0.9737, 0.9900, 0.9800, 0.0059, 87.1),
}
REF_LABELS = [('ctrl', 'ORIGINAL Lunit'), ('control', 'SimCLR control (CCSM recipe)'), ('ms', 'CCSM: MixStyle (cross-centre)'),
              ('fms', 'CCSM: MixStyle + Fourier'), ('self', 'self control + HED (adjacency recipe)'), ('neighbor', 'adjacency R=2 + HED (main)')]

def prefill_reference():
    os.makedirs(EC.RESULTS_DIR, exist_ok=True)
    for n, (p, lo, hi, lm, ls, cid) in REFERENCE.items():
        f = f"{EC.RESULTS_DIR}/{n}.json"
        if not os.path.exists(f):
            json.dump({'model': n, 'primary_auroc': p, 'primary_ci_lo': lo, 'primary_ci_hi': hi, 'primary_balacc': float('nan'),
                       'loco_mean': lm, 'loco_std': ls, 'loco_folds': {}, 'center_id_acc': cid, 'prefilled': True}, open(f, 'w'))


import gc, shutil

def prior_results():
    n = 0
    for p in glob.glob('/kaggle/input/**/abl_eval/*.json', recursive=True):
        dst = os.path.join(EC.RESULTS_DIR, os.path.basename(p))
        if not os.path.exists(dst):
            shutil.copy2(p, dst); n += 1
    if n: print(f"✓ reused {n} finished results from an earlier part")

def have_result(n): return os.path.exists(f"{EC.RESULTS_DIR}/{n}.json")
def load_meta(hf):
    return (np.array(hf['label']).astype(np.int64), np.array(hf['center']).astype(np.int64), np.array(hf['patient']).astype(np.int64))
def _ms(v):
    v = np.array(v, dtype=float); return v.mean(), (v.std(ddof=1) if len(v) > 1 else float('nan'))

def summarize():
    res = {}
    for p in glob.glob(f"{EC.RESULTS_DIR}/*.json"):
        r = json.load(open(p)); res[r['model']] = r
    per = {}
    for n, r in res.items():
        f, a, s = parse(n)
        if a != 'ctrl': per.setdefault(a, {})[s] = r
    print("\n" + "=" * 118 + f"\nLUNIT ViT-S ABLATION: {TITLE}\n(frozen linear probe; mean ± std over seeds, ddof=1)\n" + "=" * 118)
    c = res.get('lunit__ctrl')
    if c: print(f"  {'ORIGINAL Lunit':40s} LOCO {c['loco_mean']:.4f} | primary {c['primary_auroc']:.4f} | centre-ID {c['center_id_acc']:.1f}")
    rows = []
    for a, lab in [x for x in REF_LABELS if x[0] != 'ctrl'] + [(k, v) for k, v in ARM_LABELS.items()]:
        d = per.get(a)
        if not d: continue
        lm, ls = _ms([v['loco_mean'] for v in d.values()]); pm, ps = _ms([v['primary_auroc'] for v in d.values()]); cm, cs = _ms([v['center_id_acc'] for v in d.values()])
        new = '  <-- NEW' if a in ARM_LABELS else ''
        print(f"  {lab:40s} LOCO {lm:.4f} ± {ls:.4f} | primary {pm:.4f} ± {ps:.4f} | centre-ID {cm:.1f} ± {cs:.1f} | seeds {sorted(d)}   per-seed LOCO: " + " ".join(f"s{s}:{d[s]['loco_mean']:.4f}" for s in sorted(d)) + new)
        rows.append(dict(arm=a, label=lab, seeds=','.join(map(str, sorted(d))), LOCO=f"{lm:.4f} ± {ls:.4f}", primary=f"{pm:.4f} ± {ps:.4f}", centre_ID=f"{cm:.1f} ± {cs:.1f}"))
    print("\nPaired differences (x100 LOCO AUROC, same seed):")
    for a, b, label in PAIRS:
        if a in per and b in per:
            common = sorted(set(per[a]) & set(per[b]))
            if common:
                g = [100 * (per[a][s]['loco_mean'] - per[b][s]['loco_mean']) for s in common]; m, sd = _ms(g)
                print(f"  {label:56s} " + " ".join(f"s{s}:{x:+.2f}" for s, x in zip(common, g)) + f"   mean {m:+.2f} ± {sd:.2f}  positive {sum(x > 0 for x in g)}/{len(g)}")
    if c:
        print("\nGain over the ORIGINAL weights (x100 LOCO AUROC):")
        for a in ARM_LABELS:
            if a in per:
                g = [100 * (v['loco_mean'] - c['loco_mean']) for v in per[a].values()]; m, sd = _ms(g)
                print(f"  {ARM_LABELS[a]:56s} mean {m:+.2f} ± {sd:.2f}  (seeds above original: {sum(x > 0 for x in g)}/{len(g)})")
    pd.DataFrame(rows).to_csv(f"{EC.RESULTS_DIR}/ablation_summary.csv", index=False)
    print("\nReferences: Stain Aug 0.9884 | Lunit original 0.9814.  Caveat: LOCO folds 0,1,3,4 include the held-out centre's unlabeled images in pretraining; centre 2 (primary) is clean.")


def run_seeds(train_fn, hf, prep_fn=None):
    T0 = time.time(); device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_seed(42)
    y, centers, patients = load_meta(hf)
    os.makedirs(EC.FEATURE_DIR, exist_ok=True)
    prefill_reference(); prior_results()
    last_h = 0.0
    for s in SEEDS:
        names = [f'lunit__{a}_s{s}' for a in ARMS]
        if all(have_result(n) for n in names): continue
        elapsed = (time.time() - T0) / 3600
        if elapsed + max(last_h, 0.5) > BUDGET_H:
            print(f"\n⏸ time budget reached ({elapsed:.1f} h). Save this version; run again with this OUTPUT attached as an input to finish the remaining seeds."); break
        t0 = time.time()
        for a in ARMS: train_fn(a, s)
        todo = [n for n in names if not have_result(n)]
        extract_missing(hf, todo, device)
        for n in todo:
            if os.path.exists(feat_path(n)): evaluate_model(n, y, centers, patients)
        last_h = (time.time() - t0) / 3600
        print(f"  seed {s} took {last_h:.2f} h | total {(time.time()-T0)/3600:.1f} h")
    summarize()


def load_hf():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(EC.HF_DATASET); parts = [ds[k] for k in ds.keys()]
    return concatenate_datasets(parts) if len(parts) > 1 else parts[0]


def ckpt_path(arm, seed): return f"{Cfg.CKPT_DIR}/ccsm_lunit_{arm}_s{seed}_best.pt"
def find_checkpoint(fam, arm, seed):          # used by the evaluation module
    p = ckpt_path(arm, seed); return p if os.path.exists(p) else None
POOL = {}
def train_fn(arm, seed):
    if os.path.exists(ckpt_path(arm, seed)):
        print(f"  [{arm} s{seed}] checkpoint exists -> skip training"); return
    os.makedirs(Cfg.CKPT_DIR, exist_ok=True)
    t = time.time(); net, L = pretrain(arm, POOL['imgs'], POOL['cen'], seed)
    torch.save({'model_state_dict': net.state_dict(), 'arm': arm, 'seed': seed, 'epoch': Cfg.EPOCHS, 'best_val': L}, ckpt_path(arm, seed))
    print(f"  [{arm} s{seed}] saved | {(time.time() - t) / 60:.1f} min")
    del net; gc.collect(); torch.cuda.empty_cache()

if True:
    hf = load_hf()
    print(f"✓ {len(hf):,} patches")
    POOL['imgs'], POOL['cen'] = load_pool(hf)          # unlabeled pool of centres 0,1,3,4 (5000 per centre) at 96px
    run_seeds(train_fn, hf)


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

✓ extracted ['lunit__fonly_s42'] in 25.5 min
  [lunit__fonly_s42] primary 0.9856 [0.9684,0.9912] | LOCO 0.9848±0.0052 | center-ID 94.7% | 5.6 min
  seed 42 took 0.72 h | total 0.7 h
  [fonly s43] ep 1/6 loss 0.425 (chance 4.84) inst-acc 0.960 | 2.0 min
  [fonly s43] ep 2/6 loss 0.125 (chance 4.84) inst-acc 0.996 | 2.0 min
  [fonly s43] ep 3/6 loss 0.094 (chance 4.84) inst-acc 0.998 | 2.0 min
  [fonly s43] ep 4/6 loss 0.077 (chance 4.84) inst-acc 0.998 | 2.0 min
  [fonly s43] ep 5/6 loss 0.066 (chance 4.84) inst-acc 0.999 | 2.0 min
  [fonly s43] ep 6/6 loss 0.061 (chance 4.84) inst-acc 0.999 | 2.0 min
  [fonly s43] saved | 12.0 min
  [lunit__fonly_s43] loaded ccsm_lunit_fonly_s43_best.pt  (epoch 6, final loss 0.0615)


extract lunit: 1 models: 100%|██████████| 3563/3563 [25:27<00:00,  2.33it/s]


✓ extracted ['lunit__fonly_s43'] in 25.5 min
  [lunit__fonly_s43] primary 0.9842 [0.9656,0.9909] | LOCO 0.9842±0.0060 | center-ID 95.1% | 5.3 min
  seed 43 took 0.71 h | total 1.4 h
  [fonly s44] ep 1/6 loss 0.414 (chance 4.84) inst-acc 0.963 | 2.0 min
  [fonly s44] ep 2/6 loss 0.121 (chance 4.84) inst-acc 0.997 | 2.0 min
  [fonly s44] ep 3/6 loss 0.093 (chance 4.84) inst-acc 0.998 | 2.0 min
  [fonly s44] ep 4/6 loss 0.078 (chance 4.84) inst-acc 0.997 | 2.0 min
  [fonly s44] ep 5/6 loss 0.067 (chance 4.84) inst-acc 0.999 | 2.0 min
  [fonly s44] ep 6/6 loss 0.061 (chance 4.84) inst-acc 0.999 | 2.0 min
  [fonly s44] saved | 12.0 min
  [lunit__fonly_s44] loaded ccsm_lunit_fonly_s44_best.pt  (epoch 6, final loss 0.0614)


extract lunit: 1 models: 100%|██████████| 3563/3563 [25:25<00:00,  2.34it/s]


✓ extracted ['lunit__fonly_s44'] in 25.4 min
  [lunit__fonly_s44] primary 0.9826 [0.9572,0.9904] | LOCO 0.9850±0.0049 | center-ID 94.7% | 5.3 min
  seed 44 took 0.71 h | total 2.2 h

LUNIT ViT-S ABLATION: ABLATION 3 - Fourier amplitude swap only, no MixStyle (CCSM recipe)
(frozen linear probe; mean ± std over seeds, ddof=1)
  ORIGINAL Lunit                           LOCO 0.9814 | primary 0.9783 | centre-ID 97.7
  SimCLR control (CCSM recipe)             LOCO 0.9829 ± 0.0016 | primary 0.9844 ± 0.0015 | centre-ID 97.8 ± 0.3 | seeds [42, 43, 44]   per-seed LOCO: s42:0.9846 s43:0.9825 s44:0.9815
  CCSM: MixStyle (cross-centre)            LOCO 0.9863 ± 0.0007 | primary 0.9854 ± 0.0031 | centre-ID 97.6 ± 0.1 | seeds [42, 43, 44]   per-seed LOCO: s42:0.9870 s43:0.9861 s44:0.9857
  CCSM: MixStyle + Fourier                 LOCO 0.9869 ± 0.0004 | primary 0.9876 ± 0.0004 | centre-ID 94.8 ± 0.4 | seeds [42, 43, 44]   per-seed LOCO: s42:0.9870 s43:0.9864 s44:0.9872
  self control + HED (adjacency r